In [1]:
# Importation des librairies
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Importation des modèles requis
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
# Charger le dataset préparé (avec Feature Engineering)
df = pd.read_csv('../data/train_data.csv')
df.head()

,Chambres,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,LoyerMensuel_BIF,Confort_Score,Chambres_par_Superficie
0,4.0,192.0,277.0,Rohero,17.0,2600000,4.0,0.020833
1,5.0,234.0,47.0,Rohero,37.0,2600000,3.0,0.021368
2,3.0,151.0,45.0,Kinanira,23.0,1069218,3.0,0.019868
3,5.0,227.0,174.0,Gasekebuye,22.0,2600000,5.0,0.022026
4,5.0,262.0,216.0,Gihosha,21.0,2085458,3.0,0.019084


In [3]:
# Définir features(X) et target(y) 
X = df.drop(columns=['LoyerMensuel_BIF'])
y = np.array(df["LoyerMensuel_BIF"]).reshape(-1, 1)
y = np.log1p(y)

In [4]:
# Train / test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Creation du preprocessor
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist() 
# "category" pour inclure les catégories pandas

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [6]:
# Definir la liste des modeles
models = {
    'Dummy (Moyenne)': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(max_iter=10000),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}
# Configuration de la Cross-Validation
cv_folds = 10 # on utilise 5 folds par defaut

# Scoring
scoring_metrics = {
    'MAE': 'neg_mean_absolute_error',
    'MSE': 'neg_mean_squared_error',
    'R2': 'r2',
}

In [7]:
results = []

for name, model in models.items():
    print(f"Entrainement du modele : {name}...")

    # Creation du pipeline combinant le preprocessor et le modele
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    # Lancement de la Cross-Validation
    cv_results = cross_validate(
        pipeline, X, y,
        cv=cv_folds,
        scoring=scoring_metrics,
        return_estimator=False
    )

    # Extraction ce calcul des metriques (moyenne et ecart-type)
    
    # Note : 'neg_mean_absolute_error' renvoie des valeurs négatives, on prend l'opposé pour obtenir le MAE positif.
    mae_mean = -cv_results['test_MAE'].mean()
    mae_std = cv_results['test_MAE'].std()

    # Calcul de RMSE à partir de MSE
    rmse_mean = np.sqrt(-cv_results['test_MSE'].mean())
    rmse_std = cv_results['test_MSE'].std()

    # Calcul du R2
    r2_mean = cv_results['test_R2'].mean()
    r2_std = cv_results['test_R2'].std()

    # Stockage des résultats
    results.append({
        'Model': name,
        'MAE_mean': mae_mean,
        'MAE_std': mae_std,
        'RMSE_mean': rmse_mean,
        'RMSE_std': rmse_std,
        'R2_mean': r2_mean,
        'R2_std': r2_std  
    })

    print(f"-> Terminé (MAE moyen: {mae_mean:,.0f} BIF | RMSE moyen: {rmse_mean:,.0f} BIF | R2: {r2_mean:.4f})\n")

# Création du DataFrame de comparaison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='RMSE_mean', ascending=True) # Trier par le meilleur RMSE

print("\n=== RÉCAPITULATIF DES MODÈLES (TRIÉ PAR RMSE) ===")
print(results_df[['Model', 'MAE_mean', 'RMSE_mean', 'R2_mean']].to_string(index=False, float_format="%.2f"))

Entrainement du modele : Dummy (Moyenne)...
-> Terminé (MAE moyen: 1 BIF | RMSE moyen: 1 BIF | R2: -0.0417)

Entrainement du modele : Linear Regression...
-> Terminé (MAE moyen: 0 BIF | RMSE moyen: 0 BIF | R2: 0.8181)

Entrainement du modele : Ridge...
-> Terminé (MAE moyen: 0 BIF | RMSE moyen: 0 BIF | R2: 0.8183)

Entrainement du modele : Lasso...
-> Terminé (MAE moyen: 1 BIF | RMSE moyen: 1 BIF | R2: -0.0417)

Entrainement du modele : Decision Tree...
-> Terminé (MAE moyen: 0 BIF | RMSE moyen: 0 BIF | R2: 0.6380)

Entrainement du modele : Random Forest...


/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change

-> Terminé (MAE moyen: 0 BIF | RMSE moyen: 0 BIF | R2: 0.7928)

Entrainement du modele : Gradient Boosting...


/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_gb.py:672: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_gb.py:672: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_gb.py:672: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?
/home/dorian/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_gb.py:672: DataConver

-> Terminé (MAE moyen: 0 BIF | RMSE moyen: 0 BIF | R2: 0.8189)


=== RÉCAPITULATIF DES MODÈLES (TRIÉ PAR RMSE) ===
            Model  MAE_mean  RMSE_mean  R2_mean
Gradient Boosting      0.17       0.27     0.82
            Ridge      0.17       0.27     0.82
Linear Regression      0.17       0.27     0.82
    Random Forest      0.19       0.29     0.79
    Decision Tree      0.25       0.38     0.64
  Dummy (Moyenne)      0.52       0.64    -0.04
            Lasso      0.52       0.64    -0.04


In [8]:
# 7. Sauvegarder les résultats dans l'Experiment Log
log_path = '../logs/experiment_log.csv'

# Si le fichier n'existe pas encore, on crée l'en-tête
try:
    existing_log = pd.read_csv(log_path)
except FileNotFoundError:
    existing_log = pd.DataFrame(columns=['Model', 'Parameters', 'MAE', 'RMSE', 'R2', 'Observations'])

# Formater les résultats pour l'ajout
new_rows = []
for _, row in results_df.iterrows():
    new_rows.append({
        'Model': row['Model'],
        'Parameters': f'Default (CV={cv_folds})', # Vous pouvez ajouter les paramètres ici si vous les changez
        'MAE': round(row['MAE_mean'], 0),
        'RMSE': round(row['RMSE_mean'], 0),
        'R2': round(row['R2_mean'], 4),
        'Observations': f"MAE Std: {row['MAE_std']:.0f}, RMSE Std: {row['RMSE_std']:.0f}, R2 Std: {row['R2_std']:.4f}"
    })

updated_log = pd.concat([existing_log, pd.DataFrame(new_rows)], ignore_index=True)
updated_log.to_csv(log_path, index=False)
print(f"Résultats sauvegardés dans {log_path}")

Résultats sauvegardés dans ../logs/experiment_log.csv
